In [0]:
# ============================================================
# Notebook: 06_customers_scd2
# Purpose : Generate customer profile changes and apply
#           SCD Type 2 logic to preserve history.
# ============================================================

from pyspark.sql.functions import col, lit, when, current_date, current_timestamp, concat
from pyspark.sql.functions import rand

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

DIM_CUSTOMERS_TABLE = "data_engineering.silver_layer.dim_customers"
CUSTOMER_CHANGES_TABLE = "data_engineering.table_metadata.customer_scd2_changes_audit"
CHANGED_CUSTOMERS_STAGE_TABLE = "data_engineering.table_metadata.changed_customers_scd2_stage"

# ------------------------------------------------------------
# 2. Read current customer dimension
#    We only take current records because SCD2 compares incoming
#    changes against the latest active customer profile.
# ------------------------------------------------------------

current_customers_df = spark.table(DIM_CUSTOMERS_TABLE) \
    .filter(col("is_current") == True)

print("Current customer records:", current_customers_df.count())


# ------------------------------------------------------------
# 3. Pick a small sample of customers to simulate profile changes
#    In real world, these changes would come from a customer
#    master source system. Here we simulate them for the project.
# ------------------------------------------------------------

customer_changes_df = current_customers_df.sample(fraction=0.01, seed=99)

print("Simulated customer changes:", customer_changes_df.count())


# ------------------------------------------------------------
# 4. Simulate changes in SCD2-tracked fields
#    We track:
#       - risk_tier
#       - address
#
#    These are the fields where history matters.
# ------------------------------------------------------------

customer_changes_df = customer_changes_df.withColumn(
    "risk_tier",
    when(col("risk_tier") == "LOW", lit("MEDIUM"))
    .when(col("risk_tier") == "MEDIUM", lit("HIGH"))
    .otherwise(lit("LOW"))
).withColumn(
    "address",
    concat(col("address"), lit("_UPDATED"))
).withColumn(
    "change_date",
    current_date()
).withColumn(
    "change_timestamp",
    current_timestamp()
)


# ------------------------------------------------------------
# 5. Save simulated incoming changes for validation/audit
# ------------------------------------------------------------

customer_changes_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(CUSTOMER_CHANGES_TABLE)

print("Customer change events saved for audit.")

customer_changes_df.display()

# ------------------------------------------------------------
# 6. Identify customers where tracked attributes changed
#    We compare incoming customer changes with current customer
#    dimension records.
#
#    IMPORTANT:
#    We materialize changed customers into a staging Delta table
#    before applying MERGE. This prevents Spark lazy evaluation
#    from recalculating changed_customers_df after old records
#    are expired.
# ------------------------------------------------------------

current_dim_df = spark.table(DIM_CUSTOMERS_TABLE) \
    .filter(col("is_current") == True) \
    .select(
        col("account_id"),
        col("risk_tier").alias("current_risk_tier"),
        col("address").alias("current_address")
    )

incoming_changes_df = spark.table(CUSTOMER_CHANGES_TABLE)

changed_customers_df = incoming_changes_df.alias("source") \
    .join(
        current_dim_df.alias("target"),
        col("source.account_id") == col("target.account_id"),
        "inner"
    ) \
    .filter(
        (col("source.risk_tier") != col("target.current_risk_tier")) |
        (col("source.address") != col("target.current_address"))
    ) \
    .select("source.*")

print("Customers with SCD2 tracked changes:", changed_customers_df.count())


# ------------------------------------------------------------
# 7. Materialize changed customers into staging table
# ------------------------------------------------------------

changed_customers_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(CHANGED_CUSTOMERS_STAGE_TABLE)

print("Changed customers staged successfully.")

spark.table(CHANGED_CUSTOMERS_STAGE_TABLE).display()


# ------------------------------------------------------------
# 8. Expire old current records
#    Existing current record becomes historical.
# ------------------------------------------------------------

spark.sql(f"""
MERGE INTO {DIM_CUSTOMERS_TABLE} AS target
USING {CHANGED_CUSTOMERS_STAGE_TABLE} AS source
ON target.account_id = source.account_id
AND target.is_current = true

WHEN MATCHED THEN UPDATE SET
    target.is_current = false,
    target.effective_end_date = source.change_date,
    target.updated_at = source.change_timestamp
""")

print("Old current customer records expired.")


# ------------------------------------------------------------
# 9. Insert new current records
#    After expiring old rows, insert the changed customer profile
#    as the new current version.
# ------------------------------------------------------------

staged_changed_customers_df = spark.table(CHANGED_CUSTOMERS_STAGE_TABLE)

new_current_records_df = staged_changed_customers_df.select(
    col("account_id"),
    col("customer_id"),
    col("full_name"),
    col("country"),
    col("risk_tier"),
    col("address"),
    col("change_date").alias("effective_start_date"),
    lit(None).cast("date").alias("effective_end_date"),
    lit(True).alias("is_current"),
    current_date().alias("created_at"),
    current_date().alias("updated_at")
)

print("New current records to insert:", new_current_records_df.count())

new_current_records_df.write.format("delta") \
    .mode("append") \
    .saveAsTable(DIM_CUSTOMERS_TABLE)

print("New current customer records inserted.")


# ------------------------------------------------------------
# 10. Validation
# ------------------------------------------------------------

print("Customer SCD2 table summary:")

spark.sql(f"""
SELECT
    COUNT(*) AS total_customer_rows,
    COUNT(DISTINCT account_id) AS distinct_accounts,
    SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) AS current_records,
    SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END) AS historical_records
FROM {DIM_CUSTOMERS_TABLE}
""").display()


